# SkillBridge: Does NLP improve OJT recommendations?

Open in Google Colab: **File -> Upload notebook**, choose this `.ipynb`, then run cells in order.
This notebook is self-contained: it embeds a versioned snapshot of the actual system's pure Python
text/preprocessing helpers and a small synthetic development dataset. It never connects to Django,
Supabase, student accounts, or production records. Downloading model resources happens only in setup.

**Research question:** Does NLP improve independently judged recommendation relevance beyond
assessment/location matching, at an acceptable computation cost?

Do not aim for a predetermined 85/90% result. The notebook reports failures and negative gains too.
The bundled 60 cases demonstrate the workflow; they cannot establish final capstone effectiveness.
If NLP does not help, that is an actionable result: investigate evidence quality and ranking assumptions.
Generated category text is derived from scores and may add no independent information.

## 1. Setup (CPU is sufficient)
Use a fresh Colab runtime. The default setup installs spaCy small; medium and Stanza are optional.
No medium word vectors are used in these TF-IDF experiments. These are preprocessing pipelines,
not three different semantic embedding algorithms. Package/model versions are exported with results.


In [ ]:
import os, sys, subprocess, platform, importlib.util
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
INSTALL_PACKAGES = IN_COLAB
INSTALL_MEDIUM = True
INSTALL_STANZA = False  # turn on to compare Stanza; resource download is larger
if INSTALL_PACKAGES:
    packages = ["numpy>=1.26,<3", "scikit-learn>=1.4,<2", "pandas>=2,<3", "matplotlib>=3.8,<4", "spacy>=3.7,<4"]
    if INSTALL_STANZA:
        packages.append("stanza>=1.8,<2")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    for package in ["en_core_web_sm"] + (["en_core_web_md"] if INSTALL_MEDIUM else []):
        if importlib.util.find_spec(package) is None:
            subprocess.check_call([sys.executable, "-m", "spacy", "download", package])
    if INSTALL_STANZA:
        import stanza
        stanza.download("en", processors="tokenize,lemma", verbose=False)
print("Python:", sys.version)
print("Runtime:", platform.platform(), "CPU:", platform.processor())
if IN_COLAB:
    print(subprocess.run(["lscpu"], capture_output=True, text=True).stdout)
    print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)


## 2. Unpack the embedded system snapshot
This pins the helpers used in this study. New system changes require a newly versioned notebook;
do not silently replace helpers after viewing the final test. Source hashes are recorded.
The engine reproduces per-student TF-IDF and the 60/25/15 formula, but it is an offline computation
benchmark: authorization, batch gates, saved-profile freshness, capacity, and placement approval
remain separate application behaviors. Rounded ties follow the supplied candidate order.


In [ ]:
from pathlib import Path
import json, hashlib, sys
WORK = Path("skillbridge_nlp_study").resolve()
WORK.mkdir(exist_ok=True)
EMBEDDED_FILES = {'evaluation_engine.py': '"""Offline SkillBridge experiments. No Django, database writes, or model downloads.\n\nThe sibling sb_snapshot package is a verbatim snapshot of production helpers.\nLabels are consumed only by metrics/diagnostics, never by build_inputs/rank.\n"""\nfrom __future__ import annotations\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport statistics\nimport time\nfrom pathlib import Path\nfrom types import SimpleNamespace as Obj\n\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nfrom sb_snapshot import recommendation_nlp as nlp\nfrom sb_snapshot import skill_taxonomy\n\nSEED = 20261018\nWEIGHTS = (0.60, 0.25, 0.15)\nDEFAULT_METHODS = [\'structured_only\', \'tfidf_regex\', \'spacy_sm\', \'spacy_md\', \'stanza_en\']\n\n\nclass Items(list):\n    def all(self):\n        return self\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef environment():\n    versions = {}\n    for name in [\'numpy\', \'scikit-learn\', \'spacy\', \'stanza\', \'en-core-web-sm\', \'en-core-web-md\']:\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = \'not installed\'\n    return {\'python\': platform.python_version(), \'platform\': platform.platform(),\n            \'cpu\': platform.processor(), \'logical_cpu_count\': __import__(\'os\').cpu_count(),\n            \'packages\': versions, \'seed\': SEED, \'weights\': list(WEIGHTS),\n            \'text_generation_version\': nlp.TEXT_GENERATION_VERSION,\n            \'note\': \'Record RAM/CPU model from the Colab environment cell too. CPU pipelines; no GPU comparison.\'}\n\n\ndef validate(data):\n    required = {\'metadata\', \'categories\', \'positions\', \'cases\'}\n    if not required <= data.keys():\n        raise ValueError(f\'Missing dataset sections: {required - data.keys()}\')\n    for section in [\'categories\', \'positions\', \'cases\']:\n        rows = data[section]\n        if not rows or len({str(r[\'id\']) for r in rows}) != len(rows):\n            raise ValueError(f\'{section}: require nonempty rows and unique IDs\')\n    if any(not isinstance(p[\'id\'], int) for p in data[\'positions\']):\n        raise ValueError(\'Position IDs must be integers, matching production IDs.\')\n    names = [c[\'name\'] for c in data[\'categories\']]\n    if len(set(names)) != len(names):\n        raise ValueError(\'Category names must be unique\')\n    ids = {p[\'id\'] for p in data[\'positions\']}\n    if len(ids) < 3:\n        raise ValueError(\'At least 3 candidate positions are required for @3 metrics\')\n    groups = {\'dev\': set(), \'test\': set()}\n    for row in data[\'positions\'] + data[\'cases\']:\n        values = row[\'requirements\'] if \'requirements\' in row else row[\'scores\']\n        if set(values) != set(names):\n            raise ValueError(\'Every score/requirement vector must cover exactly the category names\')\n        if any(isinstance(v, bool) or not isinstance(v, (int, float)) or\n               not np.isfinite(v) or not 0 <= v <= 100 for v in values.values()):\n            raise ValueError(\'Scores and requirements must be finite percentages in [0,100]\')\n        if not any(values.values()):\n            raise ValueError(\'All-zero profiles/requirements are excluded by production; remove and report them\')\n    for p in data[\'positions\']:\n        coords = (p.get(\'lat\'), p.get(\'lng\'))\n        if (coords[0] is None) != (coords[1] is None):\n            raise ValueError(\'Supply both position coordinates or neither\')\n        if coords[0] is not None and not (-90 <= coords[0] <= 90 and -180 <= coords[1] <= 180):\n            raise ValueError(\'Invalid position coordinates\')\n    for c in data[\'cases\']:\n        if c[\'split\'] not in groups or not c.get(\'group_id\'):\n            raise ValueError(\'Each case needs split dev/test and a pseudonymous student group_id\')\n        groups[c[\'split\']].add(c[\'group_id\'])\n        coordinates = nlp.extract_coordinates(c.get(\'address\', {}))\n        if coordinates is not None and (not all(np.isfinite(v) for v in coordinates) or\n                                       not -90 <= coordinates[0] <= 90 or\n                                       not -180 <= coordinates[1] <= 180):\n            raise ValueError(\'Invalid student coordinates\')\n        if c[\'expected_position_id\'] not in ids:\n            raise ValueError(\'Expected position must occur in the candidate catalog\')\n        if \'relevant_position_ids\' in c:\n            relevant = c[\'relevant_position_ids\']\n            if not relevant or len(set(relevant)) != len(relevant) or not set(relevant) <= ids:\n                raise ValueError(\'Relevance judgments must be nonempty, unique, and in the candidate catalog\')\n            if c[\'expected_position_id\'] not in relevant:\n                raise ValueError(\'Expected first-choice role must be included among relevant positions\')\n    if groups[\'dev\'] & groups[\'test\']:\n        raise ValueError(\'A student/group occurs in both dev and test: split groups before evaluation\')\n    # Repeated profiles across splits are flagged, not silently treated as independent evidence.\n    vectors = {split: {fingerprint(c[\'scores\']) for c in data[\'cases\'] if c[\'split\'] == split}\n               for split in groups}\n    return {\'duplicate_score_vectors_across_splits\': len(vectors[\'dev\'] & vectors[\'test\']),\n            \'missing_student_coordinates\': sum(nlp.extract_coordinates(c.get(\'address\', {})) is None\n                                               for c in data[\'cases\']),\n            \'missing_position_coordinates\': sum(p.get(\'lat\') is None for p in data[\'positions\'])}\n\n\ndef build_inputs(data, split):\n    validate(data)\n    categories = [Obj(**c) for c in data[\'categories\']]\n    positions = []\n    for p in data[\'positions\']:\n        reqs = Items(Obj(skill_category=c, required_percentage=p[\'requirements\'][c.name]) for c in categories)\n        positions.append(Obj(id=p[\'id\'], title=p[\'title\'], tags=p.get(\'tags\', []),\n                             company=Obj(name=p.get(\'company\', \'Synthetic company\')), requirements=reqs))\n    cases = [c for c in data[\'cases\'] if c[\'split\'] == split]\n    if not cases:\n        raise ValueError(f\'No {split} cases\')\n    # Explicit allowlist: no label, archetype, expected role, split, or approval text enters generators.\n    profiles = []\n    for c in cases:\n        scores = [Obj(skill_category=cat, percentage=c[\'scores\'][cat.name]) for cat in categories]\n        profiles.append(nlp.generate_competency_insights(scores)[\'competency_profile_text\'])\n    descriptions = [nlp.build_position_description(p) for p in positions]\n    students = np.array([[c[\'scores\'][cat.name] / 100 for cat in categories] for c in cases])\n    requirements = np.array([[p[\'requirements\'][cat.name] / 100 for cat in categories]\n                             for p in data[\'positions\']])\n    category = cosine_similarity(students, requirements)\n    location = np.array([[nlp.location_similarity(c.get(\'address\', {}), p.get(\'lat\'), p.get(\'lng\'))[0]\n                          for p in data[\'positions\']] for c in cases])\n    return {\'cases\': cases, \'positions\': data[\'positions\'], \'profiles\': profiles,\n            \'descriptions\': descriptions, \'category\': category, \'location\': location,\n            \'input_hash\': fingerprint({\'profiles\': profiles, \'descriptions\': descriptions,\n                                       \'category\': category.tolist(), \'location\': location.tolist()})}\n\n\ndef preprocess(texts, method):\n    if method == \'tfidf_regex\':\n        return nlp._simple_preprocess(texts)\n    processed, actual, reason = nlp.preprocess_texts(texts, method, allow_fallback=False)\n    if actual != method or reason:\n        raise ValueError(\'Requested model unavailable: no fallback metrics allowed\')\n    return processed\n\n\ndef text_scores(profiles, descriptions):\n    # Production scope: fit a separate vectorizer for each student + all candidates.\n    scores = []\n    for profile in profiles:\n        try:\n            vectors = TfidfVectorizer(ngram_range=(1, 3), min_df=1,\n                                      sublinear_tf=True).fit_transform([profile] + descriptions)\n            scores.append(cosine_similarity(vectors[:1], vectors[1:])[0])\n        except ValueError as exc:\n            if \'empty vocabulary\' not in str(exc):\n                raise\n            scores.append(np.zeros(len(descriptions)))\n    return np.asarray(scores)\n\n\ndef rank(scores):\n    # Production sorts stored percentage scores rounded to 2 decimal places.\n    # Preserve supplied candidate order for rounded ties; DB query order may differ.\n    rounded = np.array([[round(float(value) * 100, 2) for value in row] for row in scores])\n    return np.argsort(-rounded, axis=1, kind=\'stable\')\n\n\ndef metrics(inputs, scores):\n    cases, positions = inputs[\'cases\'], inputs[\'positions\']\n    ids = [p[\'id\'] for p in positions]\n    order = rank(scores)\n    expected = [c[\'expected_position_id\'] for c in cases]\n    predictions = [ids[row[0]] for row in order]\n    labels = sorted(set(expected) | set(predictions))\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        expected, predictions, labels=labels, average=\'macro\', zero_division=0)\n    rows = []\n    for i, (c, indices) in enumerate(zip(cases, order)):\n        ranked_ids = [ids[j] for j in indices]\n        expected_rank = ranked_ids.index(c[\'expected_position_id\']) + 1\n        row = {\'case_id\': c[\'id\'], \'group_id\': c[\'group_id\'], \'expected\': c[\'expected_position_id\'],\n               \'predicted\': ranked_ids[0], \'expected_rank\': expected_rank,\n               \'top3_ids\': ranked_ids[:3], \'top1_correct\': int(expected_rank == 1),\n               \'hit_rate3\': int(expected_rank <= 3), \'reciprocal_rank\': 1 / expected_rank,\n               \'margin_percent\': round(float(scores[i, indices[0]]) * 100, 2) -\n                                 round(float(scores[i, indices[1]]) * 100, 2)}\n        if \'relevant_position_ids\' in c:\n            relevant = set(c[\'relevant_position_ids\'])\n            hits = [int(p in relevant) for p in ranked_ids[:3]]\n            dcg = sum(v / np.log2(j + 2) for j, v in enumerate(hits))\n            ideal = sum(1 / np.log2(j + 2) for j in range(min(3, len(relevant))))\n            row.update(precision3=sum(hits) / 3, recall3=sum(hits) / len(relevant), ndcg3=dcg / ideal)\n        rows.append(row)\n    summary = {\'top1_accuracy\': float(accuracy_score(expected, predictions)),\n               \'macro_precision\': float(precision), \'macro_recall\': float(recall), \'macro_f1\': float(f1),\n               \'hit_rate3\': float(np.mean([r[\'hit_rate3\'] for r in rows])),\n               \'mrr\': float(np.mean([r[\'reciprocal_rank\'] for r in rows]))}\n    if all(\'ndcg3\' in r for r in rows):\n        summary.update({key: float(np.mean([r[key] for r in rows]))\n                        for key in [\'precision3\', \'recall3\', \'ndcg3\']})\n    # Report every catalog role (including zero support) for diagnosis.\n    per = precision_recall_fscore_support(expected, predictions, labels=ids, zero_division=0)\n    per_role = [{\'position_id\': id_, \'title\': positions[i][\'title\'],\n                 \'precision\': float(per[0][i]), \'recall\': float(per[1][i]),\n                 \'f1\': float(per[2][i]), \'support\': int(per[3][i])} for i, id_ in enumerate(ids)]\n    return summary, rows, {\'position_ids\': ids, \'confusion_matrix\': confusion_matrix(\n        expected, predictions, labels=ids).tolist(), \'per_role\': per_role}\n\n\ndef run_method(inputs, method, repeats=3):\n    if repeats < 1:\n        raise ValueError(\'Use at least one timed repeat\')\n    trials = []\n    for repeat in range(-1, repeats):  # first untimed warm-up, including model loading\n        start = time.perf_counter()\n        if method == \'structured_only\':\n            text = np.zeros_like(inputs[\'category\'])\n            preprocessing_end = time.perf_counter()\n        else:\n            processed = preprocess(inputs[\'profiles\'] + inputs[\'descriptions\'], method)\n            preprocessing_end = time.perf_counter()\n            text = text_scores(processed[:len(inputs[\'cases\'])], processed[len(inputs[\'cases\']):])\n        total = .60 * inputs[\'category\'] + .25 * text + .15 * inputs[\'location\']\n        rank(total)\n        end = time.perf_counter()\n        if repeat >= 0:\n            trials.append({\'repeat\': repeat + 1, \'preprocessing_seconds\': preprocessing_end - start,\n                           \'feature_scoring_ranking_seconds\': end - preprocessing_end,\n                           \'total_seconds\': end - start})\n    # Category/location arrays were precomputed: add no end-to-end latency claim.\n    summary, rows, diagnostic = metrics(inputs, total)\n    nlp_summary = metrics(inputs, text)[0] if method != \'structured_only\' else None\n    for row, cat, txt, loc, scores in zip(rows, inputs[\'category\'], text, inputs[\'location\'], total):\n        row.update(category_scores=cat.tolist(), nlp_scores=txt.tolist(),\n                   location_scores=loc.tolist(), final_scores=scores.tolist())\n    return {\'method\': method, \'status\': \'available\', \'hybrid\': summary, \'nlp_only\': nlp_summary,\n            \'diagnostic\': diagnostic, \'predictions\': rows, \'timing_trials\': trials,\n            \'median_batch_seconds\': statistics.median(t[\'total_seconds\'] for t in trials),\n            \'note\': \'Structured-only removes NLP without changing relative category/location weights. \'\n                    \'Its scores max at .75; only ranking metrics are compared. \'\n                    \'Timed work excludes profile generation, category/location construction, model load, \'\n                    \'data reads, metrics, DB and network. This is a partial warm batch measurement.\'}\n\n\ndef compare(data, split=\'dev\', methods=None, repeats=3):\n    inputs = build_inputs(data, split)\n    results = []\n    for method in methods or DEFAULT_METHODS:\n        if method not in DEFAULT_METHODS:\n            raise ValueError(f\'Unknown method: {method}\')\n        print(f\'{split}: {method}\', flush=True)\n        try:\n            results.append(run_method(inputs, method, repeats))\n        except (ImportError, OSError, RuntimeError, ValueError) as exc:\n            results.append({\'method\': method, \'status\': \'unavailable_or_error\',\n                            \'error\': f\'{type(exc).__name__}: {exc}\'})\n    return {\'split\': split, \'case_count\': len(inputs[\'cases\']),\n            \'position_count\': len(inputs[\'positions\']), \'dataset_hash\': fingerprint(data),\n            \'input_hash\': inputs[\'input_hash\'], \'environment\': environment(),\n            \'dataset_metadata\': data[\'metadata\'], \'warnings\': validate(data), \'results\': results,\n            \'candidate_order\': [p[\'id\'] for p in inputs[\'positions\']]}\n\n\ndef paired_delta(candidate, baseline, key=\'top1_correct\', samples=2000):\n    a, b = candidate[\'predictions\'], baseline[\'predictions\']\n    if [r[\'case_id\'] for r in a] != [r[\'case_id\'] for r in b]:\n        raise ValueError(\'Paired comparison requires identical case order\')\n    if any(key not in r for r in a + b):\n        raise ValueError(f\'{key} requires complete relevance judgments\')\n    groups = sorted({r[\'group_id\'] for r in a})\n    diffs = np.array([a[i][key] - b[i][key] for i in range(len(a))])\n    group_indices = [[i for i, r in enumerate(a) if r[\'group_id\'] == g] for g in groups]\n    rng = np.random.default_rng(SEED)\n    boot = [float(diffs[np.concatenate([group_indices[j] for j in rng.integers(\n        0, len(groups), len(groups))])].mean()) for _ in range(samples)]\n    return {\'metric\': key, \'delta\': float(diffs.mean()),\n            \'ci95\': [float(x) for x in np.quantile(boot, [.025, .975])],\n            \'group_count\': len(groups), \'note\': \'Paired group bootstrap; synthetic/related templates \'\n                    \'still limit generalization. A CI is not a real-world guarantee.\'}\n\n\ndef lock_selection(data, dev_report, method, protocol, path):\n    if dev_report[\'split\'] != \'dev\' or dev_report[\'dataset_hash\'] != fingerprint(data):\n        raise ValueError(\'Select using the matching development report only\')\n    candidates = [r for r in dev_report[\'results\'] if r[\'method\'] == method and r[\'status\'] == \'available\']\n    if not candidates or method == \'structured_only\':\n        raise ValueError(\'Select an available NLP candidate\')\n    if protocol.get(\'primary_metric\') not in [\'top1_correct\', \'ndcg3\']:\n        raise ValueError(\'Primary metric must be top1_correct or ndcg3\')\n    for key in [\'min_top1\', \'min_macro_f1\', \'min_primary_gain\']:\n        value = protocol.get(key)\n        if value is not None and (isinstance(value, bool) or not isinstance(value, (int, float)) or\n                                  not 0 <= value <= 1):\n            raise ValueError(f\'{key}: use a fraction in [0,1] or None\')\n    selection = {\'method\': method, \'weights\': list(WEIGHTS), \'tfidf_scope\': \'per_student_plus_candidates\',\n                 \'protocol\': protocol, \'dataset_hash\': fingerprint(data), \'engine_hash\':\n                 hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                 \'helper_hash\': hashlib.sha256(Path(nlp.__file__).read_bytes()).hexdigest(),\n                 \'taxonomy_hash\': hashlib.sha256(Path(skill_taxonomy.__file__).read_bytes()).hexdigest(),\n                 \'environment\': environment(), \'dev_metrics\': candidates[0][\'hybrid\']}\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != selection:\n            raise ValueError(\'Selection already locked. Do not tune against test results; start a new study/cohort.\')\n    else:\n        path.write_text(json.dumps(selection, indent=2), encoding=\'utf-8\')\n    return selection\n\n\ndef final_test(data, selection, output_dir, repeats=3):\n    if selection[\'dataset_hash\'] != fingerprint(data):\n        raise ValueError(\'Dataset changed after lock. Freeze the complete dev/test dataset before selecting.\')\n    if selection[\'engine_hash\'] != hashlib.sha256(Path(__file__).read_bytes()).hexdigest() or \\\n       selection[\'helper_hash\'] != hashlib.sha256(Path(nlp.__file__).read_bytes()).hexdigest() or \\\n       selection[\'taxonomy_hash\'] != hashlib.sha256(Path(skill_taxonomy.__file__).read_bytes()).hexdigest():\n        raise ValueError(\'Code changed after selection; do not silently change the final method\')\n    if selection[\'environment\'][\'packages\'] != environment()[\'packages\']:\n        raise ValueError(\'Package/model versions changed after selection; reproduce the frozen environment\')\n    if not any(c[\'split\'] == \'test\' for c in data[\'cases\']):\n        return {\'status\': \'not_run\', \'reason\': \'Bundled cases are development only. Supply an independent test cohort.\'}\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    receipt = output / \'final_test.json\'\n    if receipt.exists():\n        saved = json.loads(receipt.read_text())\n        if saved[\'selection_hash\'] != fingerprint(selection):\n            raise ValueError(\'This study already has a final test for another selection\')\n        return saved\n    report = compare(data, \'test\', [\'structured_only\', selection[\'method\']], repeats)\n    available = {r[\'method\']: r for r in report[\'results\'] if r[\'status\'] == \'available\'}\n    if not {\'structured_only\', selection[\'method\']} <= available.keys():\n        return {\'status\': \'not_run\', \'reason\': \'A required method failed; inspect errors\', \'report\': report}\n    candidate, baseline = available[selection[\'method\']], available[\'structured_only\']\n    delta = paired_delta(candidate, baseline, selection[\'protocol\'][\'primary_metric\'])\n    protocol = selection[\'protocol\']\n    independent = (data[\'metadata\'].get(\'evidence_kind\') == \'expert_independent\' and\n                   bool(data[\'metadata\'].get(\'labeling_protocol\')))\n    thresholds = all(protocol.get(k) is not None for k in [\'min_top1\', \'min_macro_f1\', \'min_primary_gain\'])\n    agreed = bool(protocol.get(\'agreed_by\') and protocol.get(\'agreed_before_test_date\'))\n    if not independent or not thresholds or not agreed:\n        decision = \'Descriptive only: independent labels and pre-agreed thresholds are required for acceptance.\'\n    else:\n        passed = (candidate[\'hybrid\'][\'top1_accuracy\'] >= protocol[\'min_top1\'] and\n                  candidate[\'hybrid\'][\'macro_f1\'] >= protocol[\'min_macro_f1\'] and\n                  delta[\'ci95\'][0] >= protocol[\'min_primary_gain\'])\n        decision = (\'Meets\' if passed else \'Does not establish\') + \' the recorded criteria for this cohort.\'\n    result = {\'status\': \'complete\', \'selection_hash\': fingerprint(selection),\n              \'report\': report, \'paired_delta\': delta, \'decision\': decision,\n              \'limitation\': \'Independence/agreement metadata are researcher declarations, not externally verified. \'\n                            \'Do not reset receipts or reuse this cohort for model selection.\'}\n    receipt.write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    return result\n\n\ndef export_report(report, directory):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    (directory / \'report.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    for result in report.get(\'results\', []):\n        if result[\'status\'] != \'available\':\n            continue\n        for name, rows in [(\'predictions\', result[\'predictions\']), (\'timing\', result[\'timing_trials\']),\n                           (\'per_role\', result[\'diagnostic\'][\'per_role\'])]:\n            keys = sorted(set().union(*(r.keys() for r in rows)))\n            with (directory / f\'{result["method"]}_{name}.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n                writer = csv.DictWriter(f, fieldnames=keys)\n                writer.writeheader()\n                writer.writerows([{k: json.dumps(v) if isinstance(v, (list, dict)) else v\n                                   for k, v in r.items()} for r in rows])\n\n\ndef improvement_rows(report, method):\n    selected = next(r for r in report[\'results\'] if r[\'method\'] == method and r[\'status\'] == \'available\')\n    baseline = next(r for r in report[\'results\'] if r[\'method\'] == \'structured_only\' and r[\'status\'] == \'available\')\n    return [{\'case_id\': a[\'case_id\'], \'expected\': a[\'expected\'], \'baseline_prediction\': b[\'predicted\'],\n             \'hybrid_prediction\': a[\'predicted\'], \'expected_rank\': a[\'expected_rank\'],\n             \'change\': \'helped\' if a[\'top1_correct\'] > b[\'top1_correct\'] else\n                       \'hurt\' if a[\'top1_correct\'] < b[\'top1_correct\'] else \'unchanged\',\n             \'margin_percent\': a[\'margin_percent\']}\n            for a, b in zip(selected[\'predictions\'], baseline[\'predictions\'])]\n', 'development_dataset.json': '{\n  "metadata": {\n    "name": "SkillBridge current-helper development demonstration",\n    "evidence_kind": "synthetic_development",\n    "labeling_protocol": "Original synthetic archetype labels; not independently judged.",\n    "source_dataset_sha256": "f11af215818563d72ef8d9bef367c922a11a8d33a16bc3cd193b5ed022cba88e",\n    "note": "All 60 cases are familiar development data. No final validation is claimed. Coordinates were not present in this source: all location scores are neutral .5."\n  },\n  "categories": [\n    {\n      "id": 1,\n      "name": "Programming",\n      "tags": [\n        "Python",\n        "Java",\n        "JavaScript",\n        "OOP",\n        "algorithms",\n        "data structures",\n        "debugging",\n        "Git",\n        "problem solving",\n        "software logic"\n      ]\n    },\n    {\n      "id": 2,\n      "name": "Database",\n      "tags": [\n        "SQL",\n        "PostgreSQL",\n        "MySQL",\n        "ERD",\n        "normalization",\n        "indexing",\n        "query optimization",\n        "database design",\n        "backup",\n        "data integrity"\n      ]\n    },\n    {\n      "id": 3,\n      "name": "Networking",\n      "tags": [\n        "TCP/IP",\n        "routing",\n        "switching",\n        "subnetting",\n        "LAN",\n        "WAN",\n        "Cisco",\n        "firewall",\n        "network troubleshooting",\n        "connectivity"\n      ]\n    },\n    {\n      "id": 4,\n      "name": "Web Development",\n      "tags": [\n        "HTML",\n        "CSS",\n        "JavaScript",\n        "React",\n        "responsive design",\n        "web API",\n        "frontend",\n        "DOM",\n        "browser testing",\n        "UI development"\n      ]\n    },\n    {\n      "id": 5,\n      "name": "Cybersecurity",\n      "tags": [\n        "authentication",\n        "encryption",\n        "OWASP",\n        "firewall",\n        "access control",\n        "secure coding",\n        "vulnerability assessment",\n        "phishing",\n        "security policy"\n      ]\n    }\n  ],\n  "positions": [\n    {\n      "id": 1,\n      "title": "Backend Developer Intern",\n      "company": "ABC Software Solutions",\n      "tags": [\n        "backend",\n        "server-side",\n        "REST API",\n        "Django",\n        "Flask",\n        "authentication",\n        "database integration",\n        "business logic",\n        "API endpoint"\n      ],\n      "requirements": {\n        "Programming": 92,\n        "Database": 84,\n        "Networking": 38,\n        "Web Development": 76,\n        "Cybersecurity": 52\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 2,\n      "title": "Frontend Developer Intern",\n      "company": "ABC Software Solutions",\n      "tags": [\n        "frontend",\n        "React components",\n        "JavaScript events",\n        "responsive layout",\n        "CSS styling",\n        "browser compatibility",\n        "UI development"\n      ],\n      "requirements": {\n        "Programming": 78,\n        "Database": 38,\n        "Networking": 32,\n        "Web Development": 95,\n        "Cybersecurity": 44\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 3,\n      "title": "Database Administrator Intern",\n      "company": "DataCore Analytics",\n      "tags": [\n        "database administration",\n        "SQL optimization",\n        "schema design",\n        "backup",\n        "indexing",\n        "data validation",\n        "relational database"\n      ],\n      "requirements": {\n        "Programming": 74,\n        "Database": 96,\n        "Networking": 36,\n        "Web Development": 46,\n        "Cybersecurity": 58\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 4,\n      "title": "QA Tester Intern",\n      "company": "Tech Innovations Inc.",\n      "tags": [\n        "quality assurance",\n        "test cases",\n        "bug reporting",\n        "regression testing",\n        "manual testing",\n        "test documentation",\n        "defect tracking"\n      ],\n      "requirements": {\n        "Programming": 72,\n        "Database": 64,\n        "Networking": 42,\n        "Web Development": 66,\n        "Cybersecurity": 76\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 5,\n      "title": "Network Administrator Intern",\n      "company": "NetSecure Systems",\n      "tags": [\n        "network administration",\n        "routing",\n        "switching",\n        "subnetting",\n        "Cisco configuration",\n        "network monitoring",\n        "LAN troubleshooting"\n      ],\n      "requirements": {\n        "Programming": 42,\n        "Database": 34,\n        "Networking": 96,\n        "Web Development": 28,\n        "Cybersecurity": 78\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 6,\n      "title": "IT Support Intern",\n      "company": "XYZ Technologies",\n      "tags": [\n        "technical support",\n        "helpdesk",\n        "hardware troubleshooting",\n        "software installation",\n        "device setup",\n        "ticket handling",\n        "user assistance"\n      ],\n      "requirements": {\n        "Programming": 62,\n        "Database": 58,\n        "Networking": 84,\n        "Web Development": 52,\n        "Cybersecurity": 68\n      },\n      "lat": null,\n      "lng": null\n    }\n  ],\n  "cases": [\n    {\n      "id": "prepared-001",\n      "group_id": "prepared-001",\n      "split": "dev",\n      "scores": {\n        "Programming": 95.0,\n        "Database": 90.0,\n        "Networking": 45.0,\n        "Web Development": 80.0,\n        "Cybersecurity": 55.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-002",\n      "group_id": "prepared-002",\n      "split": "dev",\n      "scores": {\n        "Programming": 95.0,\n        "Database": 95.0,\n        "Networking": 20.0,\n        "Web Development": 80.0,\n        "Cybersecurity": 35.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-003",\n      "group_id": "prepared-003",\n      "split": "dev",\n      "scores": {\n        "Programming": 90.0,\n        "Database": 75.0,\n        "Networking": 65.0,\n        "Web Development": 80.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-004",\n      "group_id": "prepared-004",\n      "split": "dev",\n      "scores": {\n        "Programming": 100.0,\n        "Database": 100.0,\n        "Networking": 40.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-005",\n      "group_id": "prepared-005",\n      "split": "dev",\n      "scores": {\n        "Programming": 100.0,\n        "Database": 85.0,\n        "Networking": 30.0,\n        "Web Development": 75.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-006",\n      "group_id": "prepared-006",\n      "split": "dev",\n      "scores": {\n        "Programming": 90.0,\n        "Database": 90.0,\n        "Networking": 40.0,\n        "Web Development": 95.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-007",\n      "group_id": "prepared-007",\n      "split": "dev",\n      "scores": {\n        "Programming": 100.0,\n        "Database": 95.0,\n        "Networking": 25.0,\n        "Web Development": 90.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-008",\n      "group_id": "prepared-008",\n      "split": "dev",\n      "scores": {\n        "Programming": 100.0,\n        "Database": 95.0,\n        "Networking": 15.0,\n        "Web Development": 75.0,\n        "Cybersecurity": 55.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-009",\n      "group_id": "prepared-009",\n      "split": "dev",\n      "scores": {\n        "Programming": 95.0,\n        "Database": 90.0,\n        "Networking": 55.0,\n        "Web Development": 85.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-010",\n      "group_id": "prepared-010",\n      "split": "dev",\n      "scores": {\n        "Programming": 95.0,\n        "Database": 90.0,\n        "Networking": 60.0,\n        "Web Development": 80.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 1,\n      "address": {}\n    },\n    {\n      "id": "prepared-053",\n      "group_id": "prepared-053",\n      "split": "dev",\n      "scores": {\n        "Programming": 80.0,\n        "Database": 40.0,\n        "Networking": 35.0,\n        "Web Development": 90.0,\n        "Cybersecurity": 30.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-054",\n      "group_id": "prepared-054",\n      "split": "dev",\n      "scores": {\n        "Programming": 90.0,\n        "Database": 45.0,\n        "Networking": 45.0,\n        "Web Development": 85.0,\n        "Cybersecurity": 55.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-055",\n      "group_id": "prepared-055",\n      "split": "dev",\n      "scores": {\n        "Programming": 85.0,\n        "Database": 50.0,\n        "Networking": 30.0,\n        "Web Development": 95.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-056",\n      "group_id": "prepared-056",\n      "split": "dev",\n      "scores": {\n        "Programming": 90.0,\n        "Database": 85.0,\n        "Networking": 45.0,\n        "Web Development": 90.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-057",\n      "group_id": "prepared-057",\n      "split": "dev",\n      "scores": {\n        "Programming": 65.0,\n        "Database": 45.0,\n        "Networking": 30.0,\n        "Web Development": 95.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-058",\n      "group_id": "prepared-058",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 45.0,\n        "Networking": 35.0,\n        "Web Development": 85.0,\n        "Cybersecurity": 40.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-059",\n      "group_id": "prepared-059",\n      "split": "dev",\n      "scores": {\n        "Programming": 85.0,\n        "Database": 15.0,\n        "Networking": 60.0,\n        "Web Development": 100.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-060",\n      "group_id": "prepared-060",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 35.0,\n        "Networking": 50.0,\n        "Web Development": 95.0,\n        "Cybersecurity": 50.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-061",\n      "group_id": "prepared-061",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 30.0,\n        "Networking": 20.0,\n        "Web Development": 100.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-062",\n      "group_id": "prepared-062",\n      "split": "dev",\n      "scores": {\n        "Programming": 80.0,\n        "Database": 25.0,\n        "Networking": 25.0,\n        "Web Development": 95.0,\n        "Cybersecurity": 40.0\n      },\n      "expected_position_id": 2,\n      "address": {}\n    },\n    {\n      "id": "prepared-103",\n      "group_id": "prepared-103",\n      "split": "dev",\n      "scores": {\n        "Programming": 90.0,\n        "Database": 100.0,\n        "Networking": 25.0,\n        "Web Development": 30.0,\n        "Cybersecurity": 65.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-104",\n      "group_id": "prepared-104",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 85.0,\n        "Networking": 55.0,\n        "Web Development": 45.0,\n        "Cybersecurity": 50.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-105",\n      "group_id": "prepared-105",\n      "split": "dev",\n      "scores": {\n        "Programming": 95.0,\n        "Database": 90.0,\n        "Networking": 20.0,\n        "Web Development": 45.0,\n        "Cybersecurity": 45.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-106",\n      "group_id": "prepared-106",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 90.0,\n        "Networking": 45.0,\n        "Web Development": 55.0,\n        "Cybersecurity": 40.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-107",\n      "group_id": "prepared-107",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 80.0,\n        "Networking": 45.0,\n        "Web Development": 60.0,\n        "Cybersecurity": 40.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-108",\n      "group_id": "prepared-108",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 90.0,\n        "Networking": 35.0,\n        "Web Development": 20.0,\n        "Cybersecurity": 50.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-109",\n      "group_id": "prepared-109",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 100.0,\n        "Networking": 50.0,\n        "Web Development": 60.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-110",\n      "group_id": "prepared-110",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 95.0,\n        "Networking": 30.0,\n        "Web Development": 40.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-111",\n      "group_id": "prepared-111",\n      "split": "dev",\n      "scores": {\n        "Programming": 85.0,\n        "Database": 100.0,\n        "Networking": 30.0,\n        "Web Development": 35.0,\n        "Cybersecurity": 65.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-112",\n      "group_id": "prepared-112",\n      "split": "dev",\n      "scores": {\n        "Programming": 95.0,\n        "Database": 100.0,\n        "Networking": 45.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 50.0\n      },\n      "expected_position_id": 3,\n      "address": {}\n    },\n    {\n      "id": "prepared-153",\n      "group_id": "prepared-153",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 45.0,\n        "Networking": 50.0,\n        "Web Development": 90.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-154",\n      "group_id": "prepared-154",\n      "split": "dev",\n      "scores": {\n        "Programming": 80.0,\n        "Database": 55.0,\n        "Networking": 60.0,\n        "Web Development": 70.0,\n        "Cybersecurity": 75.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-155",\n      "group_id": "prepared-155",\n      "split": "dev",\n      "scores": {\n        "Programming": 60.0,\n        "Database": 80.0,\n        "Networking": 40.0,\n        "Web Development": 55.0,\n        "Cybersecurity": 75.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-156",\n      "group_id": "prepared-156",\n      "split": "dev",\n      "scores": {\n        "Programming": 80.0,\n        "Database": 55.0,\n        "Networking": 65.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-157",\n      "group_id": "prepared-157",\n      "split": "dev",\n      "scores": {\n        "Programming": 80.0,\n        "Database": 70.0,\n        "Networking": 35.0,\n        "Web Development": 75.0,\n        "Cybersecurity": 75.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-158",\n      "group_id": "prepared-158",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 65.0,\n        "Networking": 55.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 90.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-159",\n      "group_id": "prepared-159",\n      "split": "dev",\n      "scores": {\n        "Programming": 90.0,\n        "Database": 55.0,\n        "Networking": 50.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 75.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-160",\n      "group_id": "prepared-160",\n      "split": "dev",\n      "scores": {\n        "Programming": 85.0,\n        "Database": 65.0,\n        "Networking": 50.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 75.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-161",\n      "group_id": "prepared-161",\n      "split": "dev",\n      "scores": {\n        "Programming": 80.0,\n        "Database": 75.0,\n        "Networking": 50.0,\n        "Web Development": 75.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-162",\n      "group_id": "prepared-162",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 60.0,\n        "Networking": 35.0,\n        "Web Development": 65.0,\n        "Cybersecurity": 95.0\n      },\n      "expected_position_id": 4,\n      "address": {}\n    },\n    {\n      "id": "prepared-201",\n      "group_id": "prepared-201",\n      "split": "dev",\n      "scores": {\n        "Programming": 55.0,\n        "Database": 35.0,\n        "Networking": 95.0,\n        "Web Development": 20.0,\n        "Cybersecurity": 75.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-202",\n      "group_id": "prepared-202",\n      "split": "dev",\n      "scores": {\n        "Programming": 35.0,\n        "Database": 45.0,\n        "Networking": 95.0,\n        "Web Development": 30.0,\n        "Cybersecurity": 85.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-203",\n      "group_id": "prepared-203",\n      "split": "dev",\n      "scores": {\n        "Programming": 30.0,\n        "Database": 40.0,\n        "Networking": 100.0,\n        "Web Development": 40.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-204",\n      "group_id": "prepared-204",\n      "split": "dev",\n      "scores": {\n        "Programming": 40.0,\n        "Database": 50.0,\n        "Networking": 95.0,\n        "Web Development": 45.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-205",\n      "group_id": "prepared-205",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 20.0,\n        "Networking": 80.0,\n        "Web Development": 20.0,\n        "Cybersecurity": 65.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-206",\n      "group_id": "prepared-206",\n      "split": "dev",\n      "scores": {\n        "Programming": 60.0,\n        "Database": 30.0,\n        "Networking": 95.0,\n        "Web Development": 30.0,\n        "Cybersecurity": 90.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-207",\n      "group_id": "prepared-207",\n      "split": "dev",\n      "scores": {\n        "Programming": 55.0,\n        "Database": 35.0,\n        "Networking": 95.0,\n        "Web Development": 45.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-208",\n      "group_id": "prepared-208",\n      "split": "dev",\n      "scores": {\n        "Programming": 35.0,\n        "Database": 40.0,\n        "Networking": 95.0,\n        "Web Development": 60.0,\n        "Cybersecurity": 90.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-209",\n      "group_id": "prepared-209",\n      "split": "dev",\n      "scores": {\n        "Programming": 40.0,\n        "Database": 25.0,\n        "Networking": 95.0,\n        "Web Development": 45.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-210",\n      "group_id": "prepared-210",\n      "split": "dev",\n      "scores": {\n        "Programming": 40.0,\n        "Database": 60.0,\n        "Networking": 95.0,\n        "Web Development": 25.0,\n        "Cybersecurity": 85.0\n      },\n      "expected_position_id": 5,\n      "address": {}\n    },\n    {\n      "id": "prepared-251",\n      "group_id": "prepared-251",\n      "split": "dev",\n      "scores": {\n        "Programming": 60.0,\n        "Database": 60.0,\n        "Networking": 60.0,\n        "Web Development": 55.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-252",\n      "group_id": "prepared-252",\n      "split": "dev",\n      "scores": {\n        "Programming": 60.0,\n        "Database": 70.0,\n        "Networking": 85.0,\n        "Web Development": 50.0,\n        "Cybersecurity": 65.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-253",\n      "group_id": "prepared-253",\n      "split": "dev",\n      "scores": {\n        "Programming": 75.0,\n        "Database": 75.0,\n        "Networking": 95.0,\n        "Web Development": 50.0,\n        "Cybersecurity": 50.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-254",\n      "group_id": "prepared-254",\n      "split": "dev",\n      "scores": {\n        "Programming": 55.0,\n        "Database": 70.0,\n        "Networking": 85.0,\n        "Web Development": 55.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-255",\n      "group_id": "prepared-255",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 65.0,\n        "Networking": 90.0,\n        "Web Development": 40.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-256",\n      "group_id": "prepared-256",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 50.0,\n        "Networking": 85.0,\n        "Web Development": 50.0,\n        "Cybersecurity": 60.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-257",\n      "group_id": "prepared-257",\n      "split": "dev",\n      "scores": {\n        "Programming": 45.0,\n        "Database": 55.0,\n        "Networking": 80.0,\n        "Web Development": 45.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-258",\n      "group_id": "prepared-258",\n      "split": "dev",\n      "scores": {\n        "Programming": 60.0,\n        "Database": 60.0,\n        "Networking": 85.0,\n        "Web Development": 30.0,\n        "Cybersecurity": 80.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-259",\n      "group_id": "prepared-259",\n      "split": "dev",\n      "scores": {\n        "Programming": 60.0,\n        "Database": 65.0,\n        "Networking": 70.0,\n        "Web Development": 55.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    },\n    {\n      "id": "prepared-260",\n      "group_id": "prepared-260",\n      "split": "dev",\n      "scores": {\n        "Programming": 70.0,\n        "Database": 75.0,\n        "Networking": 95.0,\n        "Web Development": 60.0,\n        "Cybersecurity": 70.0\n      },\n      "expected_position_id": 6,\n      "address": {}\n    }\n  ]\n}', 'independent_dataset_template.json': '{\n  "metadata": {\n    "name": "REPLACE_WITH_STUDY_NAME",\n    "evidence_kind": "expert_independent",\n    "labeling_protocol": "REPLACE: who labeled, rubric, blinding, date, adjudication, consent/data handling"\n  },\n  "categories": [\n    {\n      "id": 1,\n      "name": "Programming",\n      "tags": [\n        "Python",\n        "Java",\n        "JavaScript",\n        "OOP",\n        "algorithms",\n        "data structures",\n        "debugging",\n        "Git",\n        "problem solving",\n        "software logic"\n      ]\n    },\n    {\n      "id": 2,\n      "name": "Database",\n      "tags": [\n        "SQL",\n        "PostgreSQL",\n        "MySQL",\n        "ERD",\n        "normalization",\n        "indexing",\n        "query optimization",\n        "database design",\n        "backup",\n        "data integrity"\n      ]\n    },\n    {\n      "id": 3,\n      "name": "Networking",\n      "tags": [\n        "TCP/IP",\n        "routing",\n        "switching",\n        "subnetting",\n        "LAN",\n        "WAN",\n        "Cisco",\n        "firewall",\n        "network troubleshooting",\n        "connectivity"\n      ]\n    },\n    {\n      "id": 4,\n      "name": "Web Development",\n      "tags": [\n        "HTML",\n        "CSS",\n        "JavaScript",\n        "React",\n        "responsive design",\n        "web API",\n        "frontend",\n        "DOM",\n        "browser testing",\n        "UI development"\n      ]\n    },\n    {\n      "id": 5,\n      "name": "Cybersecurity",\n      "tags": [\n        "authentication",\n        "encryption",\n        "OWASP",\n        "firewall",\n        "access control",\n        "secure coding",\n        "vulnerability assessment",\n        "phishing",\n        "security policy"\n      ]\n    }\n  ],\n  "positions": [\n    {\n      "id": 1,\n      "title": "Backend Developer Intern",\n      "company": "ABC Software Solutions",\n      "tags": [\n        "backend",\n        "server-side",\n        "REST API",\n        "Django",\n        "Flask",\n        "authentication",\n        "database integration",\n        "business logic",\n        "API endpoint"\n      ],\n      "requirements": {\n        "Programming": 92,\n        "Database": 84,\n        "Networking": 38,\n        "Web Development": 76,\n        "Cybersecurity": 52\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 2,\n      "title": "Frontend Developer Intern",\n      "company": "ABC Software Solutions",\n      "tags": [\n        "frontend",\n        "React components",\n        "JavaScript events",\n        "responsive layout",\n        "CSS styling",\n        "browser compatibility",\n        "UI development"\n      ],\n      "requirements": {\n        "Programming": 78,\n        "Database": 38,\n        "Networking": 32,\n        "Web Development": 95,\n        "Cybersecurity": 44\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 3,\n      "title": "Database Administrator Intern",\n      "company": "DataCore Analytics",\n      "tags": [\n        "database administration",\n        "SQL optimization",\n        "schema design",\n        "backup",\n        "indexing",\n        "data validation",\n        "relational database"\n      ],\n      "requirements": {\n        "Programming": 74,\n        "Database": 96,\n        "Networking": 36,\n        "Web Development": 46,\n        "Cybersecurity": 58\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 4,\n      "title": "QA Tester Intern",\n      "company": "Tech Innovations Inc.",\n      "tags": [\n        "quality assurance",\n        "test cases",\n        "bug reporting",\n        "regression testing",\n        "manual testing",\n        "test documentation",\n        "defect tracking"\n      ],\n      "requirements": {\n        "Programming": 72,\n        "Database": 64,\n        "Networking": 42,\n        "Web Development": 66,\n        "Cybersecurity": 76\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 5,\n      "title": "Network Administrator Intern",\n      "company": "NetSecure Systems",\n      "tags": [\n        "network administration",\n        "routing",\n        "switching",\n        "subnetting",\n        "Cisco configuration",\n        "network monitoring",\n        "LAN troubleshooting"\n      ],\n      "requirements": {\n        "Programming": 42,\n        "Database": 34,\n        "Networking": 96,\n        "Web Development": 28,\n        "Cybersecurity": 78\n      },\n      "lat": null,\n      "lng": null\n    },\n    {\n      "id": 6,\n      "title": "IT Support Intern",\n      "company": "XYZ Technologies",\n      "tags": [\n        "technical support",\n        "helpdesk",\n        "hardware troubleshooting",\n        "software installation",\n        "device setup",\n        "ticket handling",\n        "user assistance"\n      ],\n      "requirements": {\n        "Programming": 62,\n        "Database": 58,\n        "Networking": 84,\n        "Web Development": 52,\n        "Cybersecurity": 68\n      },\n      "lat": null,\n      "lng": null\n    }\n  ],\n  "cases": [\n    {\n      "id": "replace-dev-case",\n      "group_id": "replace-student-A",\n      "split": "dev",\n      "scores": {\n        "Programming": null,\n        "Database": null,\n        "Networking": null,\n        "Web Development": null,\n        "Cybersecurity": null\n      },\n      "expected_position_id": null,\n      "relevant_position_ids": [],\n      "address": {}\n    },\n    {\n      "id": "replace-test-case",\n      "group_id": "replace-student-B",\n      "split": "test",\n      "scores": {\n        "Programming": null,\n        "Database": null,\n        "Networking": null,\n        "Web Development": null,\n        "Cybersecurity": null\n      },\n      "expected_position_id": null,\n      "relevant_position_ids": [],\n      "address": {}\n    }\n  ]\n}', 'sb_snapshot/__init__.py': '', 'sb_snapshot/recommendation_nlp.py': '"""NLP, tag suggestion, rich-text, and location helpers for recommendations.\n\nAll language models are loaded lazily and cached for the life of the process.\nNo model/resource download is attempted at runtime. If an optional model is not\ninstalled, deterministic regex preprocessing keeps recommendations available.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport importlib.util\nimport os\nimport re\nfrom functools import lru_cache\nfrom pathlib import Path\n\n\nMODEL_OPTIONS = {\n    \'spacy_sm\': {\n        \'label\': \'spaCy Small\',\n        \'package\': \'en_core_web_sm\',\n    },\n    \'spacy_md\': {\n        \'label\': \'spaCy Medium\',\n        \'package\': \'en_core_web_md\',\n    },\n    \'stanza_en\': {\n        \'label\': \'Stanza English\',\n        \'package\': \'stanza English resources\',\n    },\n}\n\nfrom .skill_taxonomy import (\n    SKILL_TAGS as DEFAULT_SKILL_TAGS, ROLE_TAGS as POSITION_TAGS,\n    canonical_category, is_generic_tag, key as taxonomy_key,\n)\nfrom collections import Counter\n\nTEXT_GENERATION_VERSION = \'neutral-evidence-v2\'\n\nORIENTATION_GROUPS = [\n    (\'Backend and data-oriented\', {\'programming\', \'backend\', \'database\', \'sql\', \'python\', \'java\', \'django\', \'api\'}),\n    (\'Web and interface-oriented\', {\'web\', \'frontend\', \'design\', \'html\', \'css\', \'javascript\', \'react\', \'ui\', \'ux\'}),\n    (\'Network and infrastructure-oriented\', {\'network\', \'routing\', \'switching\', \'tcp\', \'lan\', \'wan\', \'infrastructure\', \'cisco\'}),\n    (\'Cybersecurity-oriented\', {\'security\', \'cybersecurity\', \'encryption\', \'owasp\', \'firewall\', \'authentication\'}),\n    (\'IT support and operations-oriented\', {\'support\', \'helpdesk\', \'hardware\', \'troubleshooting\', \'maintenance\', \'operations\'}),\n    (\'Data and analytics-oriented\', {\'analytics\', \'analysis\', \'statistics\', \'visualization\', \'reporting\', \'data\'}),\n]\n\nSTOP_WORDS = {\n    \'a\', \'an\', \'and\', \'are\', \'as\', \'at\', \'be\', \'by\', \'for\', \'from\', \'in\', \'is\',\n    \'it\', \'of\', \'on\', \'or\', \'that\', \'the\', \'this\', \'to\', \'with\', \'intern\', \'internship\',\n}\n\n\ndef normalize_tags(value, limit=30):\n    """Return a stable, de-duplicated list from either a list or CSV string."""\n    if value is None:\n        return []\n    values = re.split(r\'[,\\n;]+\', value) if isinstance(value, str) else value\n    if not isinstance(values, (list, tuple)):\n        return []\n    result = []\n    seen = set()\n    for item in values:\n        tag = re.sub(r\'\\s+\', \' \', str(item)).strip()\n        key = tag.casefold()\n        if tag and key not in seen:\n            result.append(tag[:80])\n            seen.add(key)\n        if len(result) >= limit:\n            break\n    return result\n\n\ndef suggest_skill_tags(name, description=\'\'):\n    canonical = canonical_category(name)\n    if canonical in DEFAULT_SKILL_TAGS:\n        return list(DEFAULT_SKILL_TAGS[canonical])\n    # Unknown categories use description evidence, not fragments of the name.\n    phrases = re.split(r\'[,;\\n]+\', str(description or \'\'))\n    return normalize_tags([phrase.strip() for phrase in phrases\n                           if 1 <= len(phrase.split()) <= 4 and\n                           not is_generic_tag(phrase, name)], limit=12)\n\n\ndef suggest_position_tags(title, requirements=None):\n    normalized = taxonomy_key(title)\n    normalized = normalized.replace(\'back end\', \'backend\').replace(\'front end\', \'frontend\')\n    normalized = normalized.replace(\'quality assurance\', \'qa tester\')\n    role_tags = []\n    for role, tags in POSITION_TAGS.items():\n        if taxonomy_key(role) in normalized:\n            role_tags.extend(tags)\n    if role_tags:\n        return normalize_tags(role_tags, limit=16)\n    # Unknown roles: a few tags from the strongest requirements, never all of\n    # every category. Admins review these suggestions before persisting them.\n    if isinstance(requirements, dict):\n        requirements = [{\'name\': name, \'percentage\': value} for name, value in requirements.items()]\n    def percentage(item):\n        try:\n            return float(item.get(\'percentage\', item.get(\'required_percentage\', 0)) or 0)\n        except (ValueError, TypeError):\n            return 0\n    suggestions = []\n    for item in sorted(requirements or [], key=percentage, reverse=True)[:2]:\n        if percentage(item) <= 0:\n            continue\n        name = item.get(\'name\') or item.get(\'category\') or \'\'\n        tags = item.get(\'tags\') or suggest_skill_tags(name)\n        suggestions.extend([t for t in normalize_tags(tags) if not is_generic_tag(t, name)][:4])\n    return normalize_tags(suggestions, limit=12)\n\n\ndef competency_label(score):\n    if score >= 90:\n        return \'excellent\'\n    if score >= 80:\n        return \'strong\'\n    if score >= 70:\n        return \'proficient\'\n    if score >= 60:\n        return \'developing\'\n    return \'basic\'\n\n\ndef score_weight(score):\n    if score >= 90:\n        return 6\n    if score >= 80:\n        return 5\n    if score >= 70:\n        return 4\n    if score >= 60:\n        return 2\n    return 1\n\n\ndef _selected_tags(tags, score, category_name=\'\'):\n    # Zero scores are not positive evidence. Retain more distinct terms in\n    # strong categories without inventing knowledge of unassessed subskills.\n    count = 8 if score >= 80 else 5 if score >= 60 else 2 if score > 0 else 0\n    return [tag for tag in normalize_tags(tags) if not is_generic_tag(tag, category_name)][:count]\n\n\ndef _evidence_parts(entries):\n    """Symmetric category evidence for students and positions, no labels/roles.\n\n    Shared tags are repeated less within this profile/requirement set. TF-IDF\n    still supplies corpus-wide inverse-document frequency. Category tags are\n    illustrative category vocabulary, not claims of separately tested skills.\n    """\n    entries = sorted(entries, key=lambda pair: (-pair[1], canonical_category(pair[0].name)))\n    selected = [(category, score, _selected_tags(category.tags, score, category.name)) for category, score in entries]\n    frequency = Counter(taxonomy_key(tag) for _, _, tags in selected for tag in set(tags))\n    fragments, evidence = [], []\n    for category, score, tags in selected:\n        name = canonical_category(category.name)\n        band = competency_label(score)\n        fragments.append(f\'{band} {name} competency ({score:g} percent); category vocabulary: {", ".join(tags) or "no specific tags"}\')\n        if score <= 0:\n            continue\n        weight = score_weight(score)\n        # Same skill + band phrase on both sides makes strengths comparable;\n        # expected positions and archetype names are never inputs here.\n        evidence.extend([f\'{name} {band} competency\'] * weight)\n        for tag in tags:\n            evidence.extend([f\'{name} {tag}\'] * max(1, weight // frequency[taxonomy_key(tag)]))\n    return fragments, evidence\n\n\ndef build_student_profile(skill_scores):\n    fragments, evidence = _evidence_parts([(s.skill_category, float(s.percentage)) for s in skill_scores])\n    if not fragments:\n        return \'No assessed competency evidence is available.\'\n    return f"The student demonstrates {\'; \'.join(fragments)}. Skill evidence: {\' \'.join(evidence)}."\n\n\ndef _category_terms(skill_score):\n    category = skill_score.skill_category\n    text = \' \'.join([category.name] + normalize_tags(category.tags)).casefold()\n    return set(re.findall(r\'[a-z][a-z0-9+#.-]*\', text))\n\n\ndef generate_competency_insights(skill_scores):\n    """Create a neutral orientation and actionable advice from assessment evidence.\n\n    The orientation never includes a company or exact position title, preventing\n    the narrative from leaking the recommendation target into NLP similarity.\n    """\n    ranked = sorted(skill_scores, key=lambda score: score.percentage, reverse=True)\n    if not ranked:\n        return {\n            \'orientation_label\': \'\',\n            \'orientation_summary\': \'\',\n            \'competency_profile_text\': \'No assessed competency evidence is available.\',\n            \'development_suggestions\': [],\n            \'supporting_categories\': [],\n        }\n\n    group_scores = []\n    for label, keywords in ORIENTATION_GROUPS:\n        weighted_score = 0.0\n        evidence_count = 0\n        for skill_score in ranked:\n            overlap = _category_terms(skill_score) & keywords\n            if overlap:\n                weighted_score += float(skill_score.percentage) * min(len(overlap), 3)\n                evidence_count += 1\n        group_scores.append((weighted_score, evidence_count, label))\n    best_score, _evidence_count, orientation_label = max(group_scores)\n    if best_score <= 0:\n        names = [score.skill_category.name for score in ranked[:2]]\n        orientation_label = f"{\' and \'.join(names)}-oriented competency profile"\n\n    leaders = ranked[:2]\n    leader_text = \' and \'.join(\n        f\'{score.skill_category.name} ({round(score.percentage, 1)}%)\'\n        for score in leaders\n    )\n    orientation_summary = (\n        f\'Assessment evidence indicates a {orientation_label.lower()} competency pattern, \'\n        f\'led by {leader_text}.\'\n    )\n\n    development_suggestions = []\n    for skill_score in sorted(ranked, key=lambda score: score.percentage)[:2]:\n        category = skill_score.skill_category\n        score = float(skill_score.percentage)\n        if score >= 80:\n            continue\n        tags = normalize_tags(category.tags)[:4]\n        focus = \', \'.join(tags) if tags else category.name.lower()\n        development_suggestions.append({\n            \'category\': category.name,\n            \'percentage\': round(score, 1),\n            \'competency_label\': competency_label(score),\n            \'message\': (\n                f\'{competency_label(score).capitalize()} competency. Strengthen {category.name.lower()} \'\n                f\'through guided practice, review activities, and hands-on projects focused on {focus}.\'\n            ),\n        })\n\n    if not development_suggestions:\n        development_suggestions.append({\n            \'category\': leaders[0].skill_category.name,\n            \'percentage\': round(float(leaders[0].percentage), 1),\n            \'competency_label\': competency_label(float(leaders[0].percentage)),\n            \'message\': \'Maintain this strong performance through increasingly complex practical projects and peer collaboration.\',\n        })\n\n    return {\n        \'orientation_label\': orientation_label,\n        \'orientation_summary\': orientation_summary,\n        \'competency_profile_text\': f\'{orientation_summary} {build_student_profile(ranked)}\',\n        \'development_suggestions\': development_suggestions,\n        \'supporting_categories\': [\n            {\n                \'category\': score.skill_category.name,\n                \'percentage\': round(float(score.percentage), 1),\n                \'competency_label\': competency_label(float(score.percentage)),\n                \'tags\': normalize_tags(score.skill_category.tags)[:6],\n            }\n            for score in ranked\n        ],\n    }\n\n\ndef build_position_description(position):\n    """Current saved requirements/tags, generated on demand without writes."""\n    requirements = list(position.requirements.all())\n    fragments, evidence = _evidence_parts([(r.skill_category, float(r.required_percentage)) for r in requirements])\n    position_tags = [t for t in normalize_tags(position.tags) if not is_generic_tag(t)]\n    # Role-specific tags remain visible but cannot overwhelm assessed skill\n    # evidence with six repetitions of unassessed role vocabulary.\n    role_evidence = [tag for tag in position_tags[:12] for _ in range(2)]\n    return (\n        f\'{position.title} at {position.company.name}. Required skill percentages: \'\n        + \'; \'.join(f\'{r.skill_category.name}: {float(r.required_percentage):g} percent\' for r in requirements)\n        + f". Competency requirements: {\'; \'.join(fragments) or \'not specified\'}. "\n        + f"Role-specific vocabulary: {\', \'.join(position_tags) or \'no saved role tags\'}. "\n        + f"Requirement evidence: {\' \'.join(evidence + role_evidence)}."\n    )\n\n\ndef _simple_preprocess(texts):\n    processed = []\n    for text in texts:\n        tokens = [\n            token.casefold() for token in re.findall(r"[A-Za-z][A-Za-z0-9+#.-]*", text)\n            if token.casefold() not in STOP_WORDS\n        ]\n        processed.append(\' \'.join(tokens))\n    return processed\n\n\n@lru_cache(maxsize=3)\ndef _load_model(model_id):\n    if model_id in (\'spacy_sm\', \'spacy_md\'):\n        import spacy\n        package = MODEL_OPTIONS[model_id][\'package\']\n        return spacy.load(package, disable=[\'parser\', \'ner\'])\n    if model_id == \'stanza_en\':\n        import stanza\n        return stanza.Pipeline(\n            \'en\', processors=\'tokenize,lemma\', tokenize_no_ssplit=True,\n            use_gpu=False, verbose=False, download_method=None,\n        )\n    raise ValueError(f\'Unsupported NLP model: {model_id}\')\n\n\ndef preprocess_texts(texts, model_id, *, allow_fallback=True):\n    """Return processed text, actual model id, and an optional fallback reason.\n\n    Evaluation passes allow_fallback=False so missing models cannot inherit\n    another model\'s metrics. Existing production callers retain fallback.\n    """\n    try:\n        model = _load_model(model_id)\n        if model_id.startswith(\'spacy_\'):\n            output = []\n            for doc in model.pipe(texts, batch_size=64):\n                output.append(\' \'.join(\n                    token.lemma_.casefold().strip()\n                    for token in doc\n                    if not token.is_stop and not token.is_punct and not token.is_space and token.lemma_.strip()\n                ))\n            return output, model_id, None\n\n        output = []\n        for text in texts:\n            doc = model(text)\n            tokens = []\n            for sentence in doc.sentences:\n                for word in sentence.words:\n                    lemma = (word.lemma or word.text).casefold().strip()\n                    if lemma and lemma.isalpha() and lemma not in STOP_WORDS:\n                        tokens.append(lemma)\n            output.append(\' \'.join(tokens))\n        return output, model_id, None\n    except Exception as exc:\n        if not allow_fallback:\n            raise\n        if model_id != \'spacy_sm\':\n            processed, actual_model, nested_reason = preprocess_texts(texts, \'spacy_sm\')\n            reason = f\'{model_id} unavailable: {exc}\'\n            if nested_reason:\n                reason = f\'{reason}; {nested_reason}\'\n            return processed, actual_model, reason\n        return _simple_preprocess(texts), \'simple_fallback\', f\'spacy_sm unavailable: {exc}\'\n\n\ndef model_status(model_id):\n    if model_id in (\'spacy_sm\', \'spacy_md\'):\n        package = MODEL_OPTIONS[model_id][\'package\']\n        available = importlib.util.find_spec(\'spacy\') is not None and importlib.util.find_spec(package) is not None\n        return {\n            \'available\': available,\n            \'error\': None if available else f\'{package} is not installed.\',\n        }\n    if model_id == \'stanza_en\':\n        resources_root = Path(os.environ.get(\'STANZA_RESOURCES_DIR\', Path.home() / \'stanza_resources\'))\n        available = importlib.util.find_spec(\'stanza\') is not None and (resources_root / \'en\').exists()\n        return {\n            \'available\': available,\n            \'error\': None if available else \'Stanza or its English resources are not installed.\',\n        }\n    return {\'available\': False, \'error\': \'Unsupported NLP model.\'}\n\n\ndef extract_coordinates(address):\n    """Read the coordinate shapes currently used by SkillBridge profiles."""\n    if not isinstance(address, dict):\n        return None\n    candidates = [\n        (address.get(\'lat\'), address.get(\'lng\')),\n        (address.get(\'latitude\'), address.get(\'longitude\')),\n        (address.get(\'pinLat\'), address.get(\'pinLng\')),\n    ]\n    pin = address.get(\'pin\')\n    if isinstance(pin, dict):\n        candidates.append((pin.get(\'lat\'), pin.get(\'lng\')))\n    for lat, lng in candidates:\n        try:\n            if lat is not None and lng is not None:\n                return float(lat), float(lng)\n        except (TypeError, ValueError):\n            continue\n    return None\n\n\ndef haversine_distance_km(lat1, lng1, lat2, lng2):\n    radius_km = 6371.0\n    lat1_rad, lng1_rad, lat2_rad, lng2_rad = map(\n        math.radians, (lat1, lng1, lat2, lng2)\n    )\n    delta_lat = lat2_rad - lat1_rad\n    delta_lng = lng2_rad - lng1_rad\n    a = (\n        math.sin(delta_lat / 2) ** 2\n        + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lng / 2) ** 2\n    )\n    return radius_km * 2 * math.asin(math.sqrt(a))\n\n\ndef location_similarity(student_address, company_lat, company_lng, max_distance_km=80.0):\n    """Return (0..1 score, distance); missing coordinates use neutral 0.5."""\n    student_coordinates = extract_coordinates(student_address)\n    if student_coordinates is None or company_lat is None or company_lng is None:\n        return 0.5, None\n    try:\n        student_lat, student_lng = student_coordinates\n        distance = haversine_distance_km(\n            float(student_lat), float(student_lng), float(company_lat), float(company_lng)\n        )\n    except (TypeError, ValueError):\n        return 0.5, None\n    return max(0.0, 1.0 - distance / max_distance_km), distance\n', 'sb_snapshot/skill_taxonomy.py': '"""Curated, reviewable IT vocabulary. Aliases normalize text, never database rows."""\nimport re\nfrom collections import Counter\nfrom difflib import SequenceMatcher\nfrom itertools import combinations\n\nTAXONOMY_VERSION = \'it-taxonomy-v2\'\nSKILL_TAGS = {\n    \'programming\': [\'Python\', \'Java\', \'JavaScript\', \'OOP\', \'algorithms\', \'data structures\', \'debugging\', \'Git\', \'problem solving\', \'software logic\'],\n    \'database management\': [\'SQL\', \'PostgreSQL\', \'MySQL\', \'ERD\', \'normalization\', \'indexing\', \'query optimization\', \'database design\', \'backup\', \'data integrity\'],\n    \'networking\': [\'TCP/IP\', \'routing\', \'switching\', \'subnetting\', \'LAN\', \'WAN\', \'Cisco\', \'firewall\', \'network troubleshooting\', \'connectivity\'],\n    \'web development\': [\'HTML\', \'CSS\', \'JavaScript\', \'React\', \'responsive design\', \'web API\', \'frontend\', \'DOM\', \'browser testing\', \'UI development\'],\n    \'cybersecurity\': [\'authentication\', \'encryption\', \'OWASP\', \'firewall\', \'access control\', \'secure coding\', \'vulnerability assessment\', \'phishing\', \'security policy\'],\n    \'cloud\': [\'AWS\', \'Azure\', \'Google Cloud\', \'cloud storage\', \'virtual machines\', \'hosting\', \'serverless\', \'containers\', \'cloud deployment\', \'cloud security\'],\n    \'devops\': [\'CI/CD\', \'Docker\', \'Kubernetes\', \'GitHub Actions\', \'deployment pipeline\', \'automation\', \'monitoring\', \'Linux server\', \'environment variables\'],\n    \'operating systems\': [\'Linux\', \'Windows\', \'Ubuntu\', \'shell\', \'command line\', \'file system\', \'processes\', \'memory management\', \'permissions\', \'system administration\'],\n    \'version control\': [\'Git\', \'GitHub\', \'GitLab\', \'branches\', \'commits\', \'merge\', \'pull request\', \'repository\', \'version history\', \'collaboration\'],\n    \'tools\': [\'VS Code\', \'Postman\', \'Figma\', \'Jira\', \'Trello\', \'API testing\', \'debugging tools\', \'documentation\', \'productivity tools\'],\n    \'design\': [\'UI design\', \'UX design\', \'wireframing\', \'prototyping\', \'visual design\', \'accessibility\'],\n    \'data analytics\': [\'data analysis\', \'statistics\', \'Python\', \'SQL\', \'visualization\', \'reporting\'],\n    \'quality assurance\': [\'test cases\', \'regression testing\', \'bug reporting\', \'test automation\', \'manual testing\', \'defect tracking\', \'test documentation\'],\n    \'technical support\': [\'helpdesk\', \'hardware troubleshooting\', \'software installation\', \'device setup\', \'ticket handling\', \'user assistance\'],\n}\nALIASES = {\n    \'database\': \'database management\', \'databases\': \'database management\', \'dbms\': \'database management\',\n    \'database administration\': \'database management\', \'cloud computing\': \'cloud\',\n    \'os\': \'operating systems\', \'operating system\': \'operating systems\',\n    \'version control systems\': \'version control\', \'git\': \'version control\',\n    \'development tools\': \'tools\', \'software tools\': \'tools\', \'developer tools\': \'tools\',\n    \'computer networking\': \'networking\', \'network administration\': \'networking\',\n    \'cyber security\': \'cybersecurity\', \'information security\': \'cybersecurity\',\n    \'software development\': \'programming\', \'computer programming\': \'programming\',\n    \'web dev\': \'web development\', \'dev ops\': \'devops\', \'qa\': \'quality assurance\',\n    \'software testing\': \'quality assurance\', \'it support\': \'technical support\',\n}\nROLE_TAGS = {\n    \'backend developer\': [\'backend\', \'server-side\', \'REST API\', \'Django\', \'Flask\', \'authentication\', \'database integration\', \'business logic\', \'API endpoint\'],\n    \'frontend developer\': [\'frontend\', \'React components\', \'JavaScript events\', \'responsive layout\', \'CSS styling\', \'browser compatibility\', \'UI development\'],\n    \'database administrator\': [\'database administration\', \'SQL optimization\', \'schema design\', \'backup\', \'indexing\', \'data validation\', \'relational database\'],\n    \'qa tester\': [\'quality assurance\', \'test cases\', \'bug reporting\', \'regression testing\', \'manual testing\', \'test documentation\', \'defect tracking\'],\n    \'network administrator\': [\'network administration\', \'routing\', \'switching\', \'subnetting\', \'Cisco configuration\', \'network monitoring\', \'LAN troubleshooting\'],\n    \'it support\': [\'technical support\', \'helpdesk\', \'hardware troubleshooting\', \'software installation\', \'device setup\', \'ticket handling\', \'user assistance\'],\n    \'cloud engineer\': [\'cloud infrastructure\', \'AWS\', \'Azure\', \'cloud deployment\', \'serverless\', \'cloud security\', \'infrastructure as code\'],\n    \'devops engineer\': [\'CI/CD\', \'Docker\', \'Kubernetes\', \'GitHub Actions\', \'deployment pipeline\', \'release automation\', \'observability\'],\n}\nGENERIC_TAGS = {\'skill\', \'skills\', \'tools\', \'tool\', \'technology\', \'technical\', \'it\', \'computer\',\n                \'development\', \'management\', \'general\', \'basic\', \'advanced\', \'operating\', \'systems\',\n                \'version\', \'control\', \'cloud\', \'devops\', \'programming\', \'database\', \'networking\'}\n\n\ndef key(value):\n    return re.sub(r\'[^a-z0-9+#]+\', \' \', str(value or \'\').casefold()).strip()\n\n\ndef canonical_category(name):\n    normalized = key(name)\n    return ALIASES.get(normalized, normalized)\n\n\ndef is_generic_tag(tag, category_name=\'\'):\n    return key(tag) in GENERIC_TAGS or (bool(category_name) and canonical_category(tag) == canonical_category(category_name))\n\n\ndef taxonomy_diagnostics(categories):\n    """Heuristic suggestions for human review, never automatic merges/updates."""\n    from .recommendation_nlp import normalize_tags\n    categories = list(categories)\n    warnings, tag_sets = [], {}\n    for index, category in enumerate(categories):\n        tags = normalize_tags(category.tags)\n        tag_sets[index] = {key(t) for t in tags}\n        codes = []\n        if not tags:\n            codes.append((\'no_tags\', \'No tags are saved.\'))\n        elif len(tags) < 4:\n            codes.append((\'few_tags\', \'Fewer than four distinct tags are saved.\'))\n        if tags and all(is_generic_tag(t, category.name) for t in tags):\n            codes.append((\'generic_tags\', \'Only generic or category-name tags are saved.\'))\n        for code, message in codes:\n            warnings.append({\'code\': code, \'category_ids\': [category.id], \'categories\': [category.name],\n                             \'message\': message, \'suggestion\': \'Review Suggest/Refresh Tags and save only appropriate tags.\'})\n    for (i, a), (j, b) in combinations(enumerate(categories), 2):\n        code = None\n        if key(a.name) == key(b.name):\n            code = \'duplicate_name\'\n        elif canonical_category(a.name) == canonical_category(b.name) or SequenceMatcher(None, key(a.name), key(b.name)).ratio() >= .88:\n            code = \'near_duplicate\'\n        if code:\n            warnings.append({\'code\': code, \'category_ids\': [a.id, b.id], \'categories\': [a.name, b.name],\n                \'message\': \'Names may describe the same competency.\',\n                \'suggestion\': \'Consider a manual consolidation plan after reviewing assessment and position references. Nothing is merged automatically.\'})\n        shared = tag_sets[i] & tag_sets[j]\n        union = tag_sets[i] | tag_sets[j]\n        if len(shared) >= 3 and len(shared) / len(union) >= .6:\n            warnings.append({\'code\': \'tag_overlap\', \'category_ids\': [a.id, b.id], \'categories\': [a.name, b.name],\n                \'message\': f\'{len(shared)} shared tags; {len(shared) / len(union):.0%} Jaccard overlap.\',\n                \'shared_tags\': sorted(shared), \'suggestion\': \'Review category boundaries and add distinguishing evidence where appropriate.\'})\n    counts = Counter(tag for tags in tag_sets.values() for tag in tags)\n    return {\'version\': TAXONOMY_VERSION, \'category_count\': len(categories), \'warnings\': warnings,\n            \'common_tags\': [{\'tag\': tag, \'category_count\': count} for tag, count in counts.most_common() if count >= 2],\n            \'note\': \'Heuristic warnings need human review. Aliases normalize generated text only; saved categories and tags remain unchanged.\'}\n', 'source_manifest.json': '{\n  "evaluation_engine.py": "3c72dd3fdfc406b803abcbfa1741539a7360bcdb71ac195b1a6625ad4358846b",\n  "development_dataset.json": "653a6c53b3812e82eeaa83b949f3174c598253fed27e2a9a82adbe9525042144",\n  "independent_dataset_template.json": "32986fdaf694233723135ec24e1a854fac0ecb198f5c6f383c7b55e0c765755d",\n  "sb_snapshot/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n  "sb_snapshot/recommendation_nlp.py": "af1f0261bff0b4a836af8633dd5aa4aa75984844674c08fe641df738c93baf4b",\n  "sb_snapshot/skill_taxonomy.py": "dfd211e2251950a07079bd3c05942abf0e847af1bb71a1f760c63bbdd22277e1",\n  "source_scoring.py": "5d640f911f169c55c072784480b3ed9e144c46ca11b486e096ddd5c75e850fdb"\n}'}
for relative, content in EMBEDDED_FILES.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text(encoding="utf-8") != content:
        raise RuntimeError(f"Snapshot file changed: {relative}. Start a new versioned study.")
    path.write_bytes(content.encode("utf-8"))
sys.path.insert(0, str(WORK))
import evaluation_engine as engine
source_manifest = json.loads((WORK / "source_manifest.json").read_text())
for relative, expected in source_manifest.items():
    if relative != "source_scoring.py":
        assert hashlib.sha256((WORK / relative).read_bytes()).hexdigest() == expected, relative
print("Snapshot verified. Workspace:", WORK)
print(json.dumps(engine.environment(), indent=2))


## 3. Choose the data and agree the protocol before evaluating
Default: **development demonstration only**. Every case is synthetic, already used in development,
and missing coordinates receive the production neutral 0.5 location score. This does not test
location discrimination. Do not label these cases as a new independent final test.

For your own study, use `independent_dataset_template.json`. Supply de-identified assessed scores,
the same candidate catalog for all methods, and `group_id` to keep all records of a student in one split.
Choose dev/test cohorts before model selection. Have instructors judge cases without seeing model
predictions: choose one expected first-choice position, and optionally mark **all** suitable positions.
Describe the rubric, assessors, disagreement/adjudication process and date in metadata.
An approved placement influenced by this recommender is not an independent relevance label.
Input only the evidence available before recommendation; no role names/labels in student text.

Real data in Colab is an external upload: use institution-approved, de-identified inputs.
The notebook will not upload local files automatically. The template deliberately has nulls until filled.

Set criteria with your adviser **before** the final test. Fractions such as .80 mean 80%, but there
is no automatic .80/.90 default. None means no acceptance claim. The chosen primary gain must have
a paired 95% bootstrap lower bound at least as large as the pre-agreed practical gain.
This is a proposed conservative study rule, not a universal rule from the literature.


In [ ]:
CUSTOM_DATASET_PATH = ""  # e.g. a manually uploaded /content/independent_dataset.json
dataset_path = Path(CUSTOM_DATASET_PATH) if CUSTOM_DATASET_PATH else WORK / "development_dataset.json"
data = json.loads(dataset_path.read_text(encoding="utf-8"))
warnings = engine.validate(data)
PROTOCOL = {
    "primary_metric": "top1_correct",  # use "ndcg3" only with complete multi-position judgments
    "min_top1": None,
    "min_macro_f1": None,
    "min_primary_gain": None,
    "agreed_by": "",
    "agreed_before_test_date": "",
    "cohort_rationale": "Describe sample size, recruitment, candidate coverage, and limits here."
}
METHODS = ["structured_only", "tfidf_regex", "spacy_sm", "spacy_md"]
if INSTALL_STANZA:
    METHODS.append("stanza_en")
REPEATS = 3
print("Dataset:", data["metadata"])
print("Cases by split:", {s: sum(c["split"] == s for c in data["cases"]) for s in ["dev", "test"]})
print("Checks:", warnings)


## 4. Check the input boundary
Permutation of expected labels must not alter the actual input text or score components.
This does not establish complete absence of bias; it verifies a concrete leakage boundary.
Category tags are vocabulary associated with a category, not proof that each technology was tested.


In [ ]:
import copy
inputs = engine.build_inputs(data, "dev")
permuted = copy.deepcopy(data)
ids = [p["id"] for p in data["positions"]]
for case in permuted["cases"]:
    case["expected_position_id"] = ids[(ids.index(case["expected_position_id"]) + 1) % len(ids)]
    case.pop("relevant_position_ids", None)
assert engine.build_inputs(permuted, "dev")["input_hash"] == inputs["input_hash"]
print("PASS: changing labels does not change text/category/location inputs.")
print("Example neutral profile:", inputs["profiles"][0])
print("Example position description:", inputs["descriptions"][0])


## 5. Development comparison: isolate contribution
- **structured_only:** assessment + location, NLP removed; relative weights are unchanged.
- **tfidf_regex:** same neutral text and TF-IDF, deterministic regex preprocessing, no trained model.
- **spaCy / Stanza:** exact snapshot preprocessing, then the same TF-IDF and hybrid formula.

NLP-only metrics are also reported for each text pipeline. Compare these separately from hybrid metrics.
All methods share cases and candidate order. Models that cannot load are reported unavailable;
no fallback silently inherits a requested model's label. One warm-up and repeated timed batches are used.
Timing excludes source/profile generation, category/location construction, model loading, metrics,
database and network; it is **not** end-to-end application latency. No result is a probability of success.


In [ ]:
import pandas as pd
dev = engine.compare(data, "dev", METHODS, REPEATS)
engine.export_report(dev, WORK / "results/dev")
rows = []
for result in dev["results"]:
    if result["status"] != "available":
        print(result["method"], result["status"], result.get("error"))
        continue
    rows.append({"method": result["method"], **result["hybrid"],
                 "median_partial_batch_seconds": result["median_batch_seconds"],
                 "nlp_only_top1": None if result["nlp_only"] is None else result["nlp_only"]["top1_accuracy"]})
development_table = pd.DataFrame(rows)
display(development_table)
print("Macro metrics classify the first predicted position; Hit Rate@3 is not Precision@3.")
print("NDCG@3 / Precision@3 / Recall@3 appear only with complete multi-position judgments.")


## 6. Diagnose the errors before changing the system
Select a candidate based on **development** results. An NLP pipeline that changes no rankings
or harms the baseline has not demonstrated added value. High Hit Rate@3 with lower Top-1 calls
for inspection of near-ties, overlapping category tags, candidate requirements, and label ambiguity.
Do not change expected labels to match model predictions.


In [ ]:
SELECTED_METHOD = "spacy_sm"  # choose after inspecting development data, before any final test
available = {r["method"]: r for r in dev["results"] if r["status"] == "available"}
if SELECTED_METHOD not in available:
    raise RuntimeError("Selected method unavailable. Install it or choose an available development candidate.")
selected = available[SELECTED_METHOD]
print("Hybrid versus no NLP:", engine.paired_delta(selected, available["structured_only"]))
if "tfidf_regex" in available:
    print("Trained preprocessor versus regex + TF-IDF:", engine.paired_delta(selected, available["tfidf_regex"]))
changes = pd.DataFrame(engine.improvement_rows(dev, SELECTED_METHOD))
display(changes[changes["change"] != "unchanged"])
display(pd.DataFrame(selected["diagnostic"]["per_role"]))
print("Confusion matrix position order:", selected["diagnostic"]["position_ids"])
display(pd.DataFrame(selected["diagnostic"]["confusion_matrix"],
                     index=selected["diagnostic"]["position_ids"], columns=selected["diagnostic"]["position_ids"]))
display(pd.DataFrame(selected["predictions"])[["case_id", "expected", "predicted", "expected_rank", "margin_percent"]])
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
development_table.plot.bar(x="method", y=["top1_accuracy", "macro_f1", "hit_rate3"], ax=axes[0], ylim=(0, 1.05))
development_table.plot.bar(x="method", y="median_partial_batch_seconds", ax=axes[1], legend=False)
axes[0].set_title("Development only: recommendation metrics")
axes[1].set_title("Partial warm batch computation (seconds)")
plt.tight_layout()
plt.savefig(WORK / "results/dev/development_comparison.png", dpi=160)
plt.show()


## 7. Freeze the choice, then evaluate the independent test once
The bundled demonstration has no final test, so this cell reports **not_run**. This is intentional.
For your own study, freeze data, method, protocol and code before running. Only the selected hybrid
and the no-NLP baseline are evaluated on test, avoiding a test-set model leaderboard.
Save/download the lock and final receipt. Colab can be reset, so these safeguards are a research
workflow aid, not tamper-proof proof of a preregistration. Never reuse a viewed test cohort for tuning.
The dataset and code hashes prevent silent edits within the same saved study.


In [ ]:
selection = engine.lock_selection(data, dev, SELECTED_METHOD, PROTOCOL, WORK / "selection_lock.json")
final = engine.final_test(data, selection, WORK / "results/final", REPEATS)
print(final.get("decision", final.get("reason")))
if final["status"] == "complete":
    print("Paired NLP gain:", final["paired_delta"])
    engine.export_report(final["report"], WORK / "results/final/details")
else:
    print("No final accuracy or acceptance claim is produced for the demonstration.")


## 8. Applying findings to SkillBridge
1. If generated descriptions overlap too much, review category/position taxonomy and assessed
   evidence with instructors. More repeated keywords are not evidence of better placements.
2. If neutral-text TF-IDF does not beat structured-only, current NLP has no demonstrated ranking
   gain under these labels. Report this and investigate on development data.
3. If spaCy does not beat regex + TF-IDF, you may have evidence for text matching but not an
   additional benefit from the trained preprocessing model. Keep those claims distinct.
4. Candidate improvements: verified question-level skill evidence, independently written
   student project descriptions, richer company requirements, or a semantic-embedding experiment.
   These require assessment/data provenance and a new versioned study. No embeddings are used here.
   Never infer individual technology mastery from an overall category percentage alone.
5. Preserve independent final labels. Choose any text/ranking changes on development data only,
   then evaluate on a new untouched cohort. Existing approved placements can be circular labels.
6. Transfer only the independently supported change, with matching helper versions, preprocessing,
   TF-IDF scope and weights. Use the exported per-case component vectors to check backend parity.
   Similarity ranks do not replace eligibility, capacity or instructor placement decisions.

The current engine uses the exact embedded text helpers and formula; parity checks still need
real production candidate ordering, current combined competency evidence and runtime model versions.
Its synthetic results must not be relabeled as live-system accuracy.

## 9. Export your study
Exports contain case-level inputs/predictions. Do not publish a ZIP with private student records.
The default bundled study is synthetic. Downloading is manual in the next cell.


In [ ]:
import shutil
(WORK / "dataset_used.json").write_text(json.dumps(data, indent=2), encoding="utf-8")
(WORK / "protocol.json").write_text(json.dumps(PROTOCOL, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(WORK.parent / "SkillBridge_NLP_Study_Results"), "zip", WORK)
print("Export ready:", archive)
DOWNLOAD_RESULTS = False  # set True when ready to download
if IN_COLAB and DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)


## Supporting sources and claim boundaries
- Gunawardana & Shani (2009), *A Survey of Accuracy Evaluation Metrics of Recommendation Tasks*,
  JMLR 10, 2935-2962. https://jmlr.org/papers/v10/gunawardana09a.html
  Supports task-appropriate metrics/experimental design, not a passing percentage.
- Qin et al. (2018), *Enhancing Person-Job Fit for Talent Recruitment: An Ability-aware Neural
  Network Approach*, SIGIR, 25-34. https://doi.org/10.1145/3209978.3210025
  Supports classification metrics in related matching; its binary outcome task differs.
- Kara et al. (2023; online 2022), *Job Recommendation Based on Extracted Skill Embeddings*,
  IntelliSys proceedings, 497-507. https://doi.org/10.1007/978-3-031-16075-2_35
  Supports skill-based matching with a TF-IDF/cosine baseline; no cross-dataset superiority claim.
- Qi et al. (2020), *Stanza: A Python Natural Language Processing Toolkit for Many Human Languages*,
  ACL System Demonstrations, 101-108. https://doi.org/10.18653/v1/2020.acl-demos.14
  Supports preprocessing comparisons, not an OJT effectiveness threshold.
- spaCy model setup: https://spacy.io/usage/models
- Stanza model setup: https://stanfordnlp.github.io/stanza/download_models.html

This notebook performs no model fine-tuning and does not establish a universal acceptable score.
A more complex model is a hypothesis to test, not evidence of capstone quality by itself.
